# KLA restoration training — one-shot Colab (clone + Run all)

Trains the residual U-Net on a **first-party synthetic, source-disjoint** corpus of 160 clean sources (768 train / 96 val / 96 test paired views). Uses only the three disclosed degradations (additive Gaussian noise, multiplicative speckle, downsampling). Results are pipeline evidence only, **not** official KLA or hidden-test scores.

This notebook is **fully automated**: it clones the public repository (no manual upload), builds the corpus, trains, freezes `models/best.pth` (the exact checkpoint the `run.py` `.npy` evaluator loads), evaluates once on held-out sources, exercises the `run.py` `.npy` contract, and downloads the trained artifacts.

**To run:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`. No prompts, no edits.

## 1. Record the actual runtime

In [ ]:
import platform, torch
print({
    'python': platform.python_version(),
    'torch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'cuda': torch.version.cuda,
})
assert torch.cuda.is_available(), 'Enable a GPU runtime: Runtime > Change runtime type > T4 GPU'
!nvidia-smi

## 2. Clone the public repository (no manual upload)

Clones the finalized submission branch directly. This is what makes the notebook one-shot.

In [ ]:
import os
REPO_URL = 'https://github.com/Veer-WebDev/kla-image-restoration.git'
BRANCH = 'kla-restoration-submission'
!rm -rf kla-image-restoration
!git clone --depth 1 --branch $BRANCH $REPO_URL
os.chdir('kla-image-restoration')
print('cwd', os.getcwd())
!git rev-parse --short HEAD
!ls

## 3. Install dependencies (keep Colab's CUDA torch)

Colab already ships a CUDA-matched PyTorch. We install everything **except** torch so the GPU build is preserved, then confirm CUDA is still available.

In [ ]:
# Install declared deps but do not touch the pre-installed CUDA torch.
!grep -viE '^\s*(torch|torchvision|torchaudio)\b' requirements.txt > _reqs_no_torch.txt
!cat _reqs_no_torch.txt
!python -m pip install -q -r _reqs_no_torch.txt
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
assert torch.cuda.is_available(), 'CUDA torch was clobbered; restart runtime and re-run.'

## 4. Construct the disclosed first-party corpus

Source split occurs by SHA-256 **before** views. Only Gaussian noise, multiplicative speckle and downsampling, across all six orders.

In [ ]:
!python scripts/generate_clean_sem_sources.py --out data/sources_big --count 160 --size 768 --seed 20260817
!python scripts/materialize_restoration_data.py --source-dir data/sources_big --out data/kla_big --seed 20260817 --views-per-source 6 --crop-size 512 --scale 2
!cat data/kla_big/dataset_card.json 2>/dev/null || true
import glob
for split in ('train','val','test'):
    n = len(glob.glob(f'data/kla_big/{split}/NoisyLR/*.npy'))
    print(split, 'pairs:', n)

## 5. Train the submission configuration

Model is selected by validation PSNR only. `data/kla_big/test` is never inspected or tuned against. On a T4 expect roughly 20–40 min.

In [ ]:
!python train.py --config configs/submission_big.yaml
!ls -la runs/kla_restoration_big_seed20260817

## 6. Freeze validation-best weights into the run.py location, evaluate held-out sources once

In [ ]:
!mkdir -p models weights results/submission_big/examples
# models/best.pth is the exact file run.py loads for the .npy evaluator contract.
!cp runs/kla_restoration_big_seed20260817/best.pth models/best.pth
!cp runs/kla_restoration_big_seed20260817/best.pth weights/final_model.pth
!cp runs/kla_restoration_big_seed20260817/resolved_config.yaml weights/final_model.config.yaml 2>/dev/null || true
!sha256sum models/best.pth | tee models/best.sha256
# LPIPS pulls a small backbone over the network (available during Colab training).
!python evaluate.py --gt-dir data/kla_big/test/GT --noisy-dir data/kla_big/test/NoisyLR --checkpoint models/best.pth --output-dir results/submission_big --save-restored results/submission_big/examples --split all --eval-mode official || python evaluate.py --gt-dir data/kla_big/test/GT --noisy-dir data/kla_big/test/NoisyLR --checkpoint models/best.pth --output-dir results/submission_big --save-restored results/submission_big/examples --split all --eval-mode official --no-lpips
!cat results/submission_big/summary.json

## 7. Exercise the evaluator-facing run.py `.npy` contract

In [ ]:
!rm -rf submission_smoke
# run.py takes positional <input-dir> <output-dir>, reads .npy recursively, writes matching .npy outputs.
!python run.py data/kla_big/test/NoisyLR submission_smoke
import numpy as np, glob, os
outs = sorted(glob.glob('submission_smoke/**/*.npy', recursive=True))
ins = sorted(glob.glob('data/kla_big/test/NoisyLR/**/*.npy', recursive=True))
print('inputs', len(ins), 'outputs', len(outs))
a = np.load(outs[0])
print('sample out', os.path.basename(outs[0]), a.shape, a.dtype, float(a.min()), float(a.max()), 'finite', bool(np.isfinite(a).all()))
assert len(ins) == len(outs), 'filename parity failed'
assert a.ndim in (2, 3) and float(a.min()) >= 0.0 and float(a.max()) <= 1.0 and bool(np.isfinite(a).all()), 'output invariants failed'
print('run.py .npy contract OK')

## 8. Archive artifacts and the trained checkpoint

Downloads a zip containing `models/best.pth` (drop it into the repo at the same path to update the submission), the evaluation summary, and manifests.

In [ ]:
import hashlib, pathlib, datetime
artifact = pathlib.Path('kla_restoration_big_artifacts.zip')
!rm -f $artifact
!zip -qr $artifact models/best.pth models/best.sha256 weights results/submission_big submission_smoke data/kla_big/dataset_card.json data/kla_big/train_manifest.csv data/kla_big/val_manifest.csv data/kla_big/test_manifest.csv
print({'artifact': str(artifact), 'sha256': hashlib.sha256(artifact.read_bytes()).hexdigest(), 'utc_finished': datetime.datetime.now(datetime.UTC).isoformat()})
try:
    from google.colab import files
    files.download(str(artifact))
except Exception as exc:
    print('auto-download unavailable, find the zip in the file browser:', exc)